In [1]:
import os
import csv
import subprocess

import pandas as pd

In [2]:
base_dir = "/home/omen/Documents/Megha/ABIDE_DATA/ABIDE_I"

output_dir = "/home/omen/Documents/Megha/ABIDE1_PREPROCESSED"

# cropped_image_folder = "/home/omen/Documents/Megha/preprocessed_ixi1/cropped"
bet_folder_loc ='/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/bet'
bias_field_corrected_loc = '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/bias_field_corrected'
flirt_registered__loc= '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/linear_registered'
affine_mat__loc = '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/affine'
warp_coeff_loc ='/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/warp'
non_linear_loc =  '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/non_linear_registered'
# complete_preprocessed_loc = '/home/omen/Documents/Megha/ABIDE1_PREPROCESSED/completely_preprocessed'

stages = {
    "bet":        bet_folder_loc,
    "bias_field": bias_field_corrected_loc,
    "flirt":      flirt_registered__loc,
    "affine":     affine_mat__loc,
    "warp":       warp_coeff_loc,
    "non_linear": non_linear_loc
    # "final_bet":  complete_preprocessed_loc,
    # "cropped_image" : cropped_image_folder
}


config_file = "T1_2_MNI152_2mm"
FSL_STANDARD_BRAIN = '/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm.nii.gz'
FSL_STANDARD_BRAIN_MASKED = '/home/omen/fsl5.0.10/data/standard/MNI152_T1_2mm_brain.nii.gz'
f_value= 0.4


In [3]:
file = pd.read_csv('/home/omen/Documents/Megha/ABIDE_I_relative_paths_with_age_sex.csv')
print(file.columns)
for rel_path in file['relative_path']:
    site = rel_path.split("/")[0]
    # site = os.path.basename(rel_path)
    # site = os.path.basename(rel_path).replace(".nii.gz", "")
    # t1_files = file[file['relative_path'].str.contains("T1", case=False)]
    print(site)

for rel_path in file['relative_path']:
    site = rel_path.split("/")[0]
    # print(site)
    out_fname = f"{site}.nii.gz" 
    print(out_fname)

FileNotFoundError: [Errno 2] No such file or directory: '/home/omen/Documents/Megha/ABIDE_I_relative_paths_with_age_sex.csv'

In [ ]:

# Loop through the filtered T1 file paths
for rel_path in file['relative_path']:
    site = rel_path.split("/")[0]
    out_fname = f"{site}.nii.gz"  # ABIDE_KKI_29473.nii.gz
    paths = {
            "bet":        os.path.join(stages["bet"], out_fname),
            "bias_field" : os.path.join(stages["bias_field"],site),
            "flirt":      os.path.join(stages["flirt"],out_fname),
            "affine":     os.path.join(stages["affine"], f"{site}.mat"),
            "warp":       os.path.join(stages["warp"],  out_fname),
            "non_linear": os.path.join(stages["non_linear"],out_fname)
            # "final_bet":  os.path.join(stages["final_bet"], out_fname),
            # "cropped_image": os.path.join(stages["cropped_image"],out_fname)
        }
    for stage, p in paths.items():
        if stage== "bias_field":
            os.makedirs(os.path.dirname(p), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(p), exist_ok=True)
    input_image = os.path.join(base_dir, rel_path)
    print(input_image)
    
    try:
        # print("cropping image")
        # subprocess.run(["robustfov", "-i", input_image, "-r", paths["cropped_image"]], check=True)

        print(f"running bet for {input_image}")
        subprocess.run([
            "bet",
            input_image,
            paths["bet"],
            "-f", str(f_value),
            "-g", "0",
            "-R"  # robust brain center estimate
             
                ], check=True)
        if not os.path.exists(paths["bet"]):
            print(f" BET did not produce output for: {input_image}")
            continue
        print("running_bias_field_correction")
        bias_base = paths["bias_field"]  
        print("-> running FAST (bias correction), output base:", bias_base)
        subprocess.run([
            "fast",
            "-B",           # output bias corrected image
            "-t", "1",      # image type = T1
            "-o", bias_base,  # output base name
            paths["bet"]
        ],check=True)
        bias_restore = bias_base + "_restore.nii.gz"
        if not os.path.exists(bias_restore):
            print("Expected FAST output not found:", bias_restore)
            print("Listing bias_field dir for debugging:", os.listdir(os.path.dirname(bias_base)))
            continue
        print("running_linear_registeration")
        
        subprocess.run([
                    "flirt",
                    "-ref", FSL_STANDARD_BRAIN_MASKED,
                    "-in", bias_restore,
                    "-omat", paths["affine"],    #change the affine - wrt the file extension
                    "-out", paths["flirt"]
                ], check=True)
        print("running non linear registeration")
        subprocess.run([
            "fnirt",
            "--in=" + bias_restore,
            "--aff=" + paths["affine"],
            "--cout=" + paths["warp"],
            "--config=" + config_file
        ], check=True)
        print("applying the transformnation")
        subprocess.run([
            "applywarp",
            "--ref=" + FSL_STANDARD_BRAIN_MASKED,
            "--in=" + bias_restore,
            "--warp=" + paths["warp"],
            "--out=" + paths["non_linear"]
        ], check=True)
        # print("lastly applying bet again on the final non linear registered image")
        # subprocess.run([
        #     "bet",
        #     paths["non_linear"],
        #     paths["final_bet"],

        # ], check=True)
        

    except subprocess.CalledProcessError as e:
                print(f"Failed processing {input_image}: {e}")
    


In [ ]:

# Loop through the filtered T1 file paths
for rel_path in file['relative_path']:
    Multiprocess(rel_path)





    def Multiprocess(rel_path, base_dir, f_value, stages, config_file, FSL_STANDARD_BRAIN_MASKED):
    
        site = rel_path.split("/")[0]
        out_fname = f"{site}.nii.gz"  # ABIDE_KKI_29473.nii.gz
        paths = {
                "bet":        os.path.join(stages["bet"], out_fname),
                "bias_field" : os.path.join(stages["bias_field"],site),
                "flirt":      os.path.join(stages["flirt"],out_fname),
                "affine":     os.path.join(stages["affine"], f"{site}.mat"),
                "warp":       os.path.join(stages["warp"],  out_fname),
                "non_linear": os.path.join(stages["non_linear"],out_fname)
                # "final_bet":  os.path.join(stages["final_bet"], out_fname),
                # "cropped_image": os.path.join(stages["cropped_image"],out_fname)
            }
        for stage, p in paths.items():
            if stage== "bias_field":
                os.makedirs(os.path.dirname(p), exist_ok=True)
            else:
                os.makedirs(os.path.dirname(p), exist_ok=True)
        input_image = os.path.join(base_dir, rel_path)
        print(input_image)
    
        try:
            # print("cropping image")
            # subprocess.run(["robustfov", "-i", input_image, "-r", paths["cropped_image"]], check=True)
    
            print(f"running bet for {input_image}")
            subprocess.run([
                "bet",
                input_image,
                paths["bet"],
                "-f", str(f_value),
                "-g", "0",
                "-R"  # robust brain center estimate
                 
                    ], check=True)
            if not os.path.exists(paths["bet"]):
                print(f" BET did not produce output for: {input_image}")
                continue
            print("running_bias_field_correction")
            bias_base = paths["bias_field"]  
            print("-> running FAST (bias correction), output base:", bias_base)
            subprocess.run([
                "fast",
                "-B",           # output bias corrected image
                "-t", "1",      # image type = T1
                "-o", bias_base,  # output base name
                paths["bet"]
            ],check=True)
            bias_restore = bias_base + "_restore.nii.gz"
            if not os.path.exists(bias_restore):
                print("Expected FAST output not found:", bias_restore)
                print("Listing bias_field dir for debugging:", os.listdir(os.path.dirname(bias_base)))
                continue
            print("running_linear_registeration")
            
            subprocess.run([
                        "flirt",
                        "-ref", FSL_STANDARD_BRAIN_MASKED,
                        "-in", bias_restore,
                        "-omat", paths["affine"],    #change the affine - wrt the file extension
                        "-out", paths["flirt"]
                    ], check=True)
            print("running non linear registeration")
            subprocess.run([
                "fnirt",
                "--in=" + bias_restore,
                "--aff=" + paths["affine"],
                "--cout=" + paths["warp"],
                "--config=" + config_file
            ], check=True)
            print("applying the transformnation")
            subprocess.run([
                "applywarp",
                "--ref=" + FSL_STANDARD_BRAIN_MASKED,
                "--in=" + bias_restore,
                "--warp=" + paths["warp"],
                "--out=" + paths["non_linear"]
            ], check=True)
            # print("lastly applying bet again on the final non linear registered image")
            # subprocess.run([
            #     "bet",
            #     paths["non_linear"],
            #     paths["final_bet"],
    
            # ], check=True)
            
    
        except subprocess.CalledProcessError as e:
                    print(f"Failed processing {input_image}: {e}")
        
